In [17]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

In [18]:
df = pd.read_csv('../data/data.csv')
df.head()

,fecha,id_producto,id_sucursal,Suma de unidades_vendidas
0,2023-01-01 00:00:00,P001,S01,56
1,2023-01-01 00:00:00,P001,S02,55
2,2023-01-01 00:00:00,P001,S03,92
3,2023-01-01 00:00:00,P001,S04,65
4,2023-01-01 00:00:00,P001,S05,66


In [19]:
# Fix columns
# Name
df = df.rename(columns={"Suma de unidades_vendidas": "ventas"})
df['ventas'] = df['ventas'].fillna(0)

# Fechas
df['fecha'] = pd.to_datetime(df['fecha'])
df['año'] = df['fecha'].dt.year

meses_num = df['fecha'].dt.month
dias_num = df['fecha'].dt.dayofweek

df['mes_sin'] = np.sin(2 * np.pi * meses_num / 12)
df['mes_cos'] = np.cos(2 * np.pi * meses_num / 12)

df['dia_sin'] = np.sin(2 * np.pi * dias_num / 7)
df['dia_cos'] = np.cos(2 * np.pi * dias_num / 7)

año_minimo = df['año'].min()
df['tendencia_anual'] = df['año'] - año_minimo

df = df.drop(columns=["fecha", "año"], errors='ignore')
df

,id_producto,id_sucursal,ventas,mes_sin,mes_cos,dia_sin,dia_cos,tendencia_anual
0,P001,S01,56,0.5,0.866025,-0.781831,0.62349,0
1,P001,S02,55,0.5,0.866025,-0.781831,0.62349,0
2,P001,S03,92,0.5,0.866025,-0.781831,0.62349,0
3,P001,S04,65,0.5,0.866025,-0.781831,0.62349,0
4,P001,S05,66,0.5,0.866025,-0.781831,0.62349,0
...,...,...,...,...,...,...,...,...
29994,P011,S02,26,-0.5,0.866025,0.781831,0.62349,0
29995,P011,S03,69,-0.5,0.866025,0.781831,0.62349,0
29996,P011,S05,33,-0.5,0.866025,0.781831,0.62349,0
29997,P012,S01,12,-0.5,0.866025,0.781831,0.62349,0


In [20]:
X_lgb = df.drop(columns=["ventas", "fecha", "año"], errors='ignore').copy()
X_lgb['id_producto'] = X_lgb['id_producto'].astype('category')
X_lgb['id_sucursal'] = X_lgb['id_sucursal'].astype('category')
y = df["ventas"]

X_train_lgb, X_test_lgb, y_train, y_test = train_test_split(
    X_lgb, y, test_size=0.2, random_state=521
)

lgbm_base = lgb.LGBMRegressor(random_state=521, n_jobs=-1)
param_grid = {
    'num_leaves': [15, 31, 50],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 500, 1000],
    'min_child_samples': [10, 20, 50]
}

grid = GridSearchCV(
    estimator=lgbm_base,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    verbose=1   
)

grid.fit(X_train_lgb, y_train)
print(f"\nThe absolute best parameters from your grid: {grid.best_params_}")

best_model = grid.best_estimator_
y_pred_grid = best_model.predict(X_test_lgb)

print(f"Grid-Optimized MAE: {mean_absolute_error(y_test, y_pred_grid):.2f} unidades de ventas")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000314 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 70
[LightGBM] [Info] Number of data points in the train set: 19199, number of used features: 6
[LightGBM] [Info] Start training from score 21.340955
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000313 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 70
[LightGBM] [Info] Number of data points in the train set: 19199, number of used features: 6
[LightGBM] [Info] Start training from score 21.274493
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000310 seconds.
You can set `force_row_wis